# Saved Plant Disease Model Tester

Use this notebook to verify `../../models/plant_disease_model.pth` and predict one or more plant leaf images without changing the main training notebook.

## 1. Import Libraries

In [1]:
import io
import json
import os

import pandas as pd
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from IPython.display import HTML, display
from PIL import Image

try:
    import ipywidgets as widgets
except ImportError:
    widgets = None
    print("ipywidgets is not installed. Use the local IMAGE_PATHS fallback cell instead.")

## 2. Model Configuration

In [2]:
TRAIN_DIR = "../../archive/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/train"
MODEL_PATH = "../../models/plant_disease_model.pth"
CLASSES_PATH = "../../models/disease_classes.json"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## 3. Define Model Architecture

In [3]:
def conv_block(in_channels, out_channels, pool=False):
    layers = [
        nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True),
    ]
    if pool:
        layers.append(nn.MaxPool2d(4))
    return nn.Sequential(*layers)


class CNNNeuralNet(nn.Module):
    def __init__(self, in_channels, num_diseases):
        super().__init__()
        self.conv1 = conv_block(in_channels, 64)
        self.conv2 = conv_block(64, 128, pool=True)
        self.res1 = nn.Sequential(conv_block(128, 128), conv_block(128, 128))
        self.conv3 = conv_block(128, 256, pool=True)
        self.conv4 = conv_block(256, 512, pool=True)
        self.res2 = nn.Sequential(conv_block(512, 512), conv_block(512, 512))
        self.classifier = nn.Sequential(
            nn.MaxPool2d(4),
            nn.Flatten(),
            nn.Linear(512, num_diseases),
        )

    def forward(self, x):
        out = self.conv1(x)
        out = self.conv2(out)
        out = self.res1(out) + out
        out = self.conv3(out)
        out = self.conv4(out)
        out = self.res2(out) + out
        return self.classifier(out)

## 4. Load Saved Model

In [4]:
if os.path.isfile(CLASSES_PATH):
    with open(CLASSES_PATH, "r", encoding="utf-8") as f:
        classes = json.load(f)
else:
    classes = sorted(
        name for name in os.listdir(TRAIN_DIR)
        if os.path.isdir(os.path.join(TRAIN_DIR, name))
    )

model = CNNNeuralNet(3, len(classes))
state_dict = torch.load(MODEL_PATH, map_location=device)
model.load_state_dict(state_dict)
model.to(device)
model.eval()

with torch.no_grad():
    test_output = model(torch.zeros(1, 3, 256, 256, device=device))

print("Saved model loaded successfully.")
print(f"Classes: {len(classes)}")
print(f"Output shape: {tuple(test_output.shape)}")

Saved model loaded successfully.
Classes: 38
Output shape: (1, 38)


## 5. Prediction Helpers

In [5]:
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
])


def predict_pil_image(image):
    image = image.convert("RGB")
    tensor = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(tensor)
        probabilities = torch.softmax(output, dim=1)[0]
        confidence, pred_idx = torch.max(probabilities, dim=0)

    return classes[pred_idx.item()], confidence.item()


def show_prediction_results(results):
    rows = []
    for item in results:
        rows.append({
            "File": item["file"],
            "Prediction": item["prediction"],
            "Confidence": f"{item['confidence'] * 100:.2f}%",
        })

    df = pd.DataFrame(rows)
    display(HTML("<h3>Prediction Results</h3>"))
    display(df.style.set_properties(**{"text-align": "left"}).hide(axis="index"))

    for item in results:
        display(HTML(f"<h4>{item['file']}</h4><p><b>Prediction:</b> {item['prediction']}<br><b>Confidence:</b> {item['confidence'] * 100:.2f}%</p>"))
        display(item["image"].resize((220, 220)))

## 6. Upload Multiple Images and Predict

Run the next cell, click **Upload**, select one or more images, then click **Predict Uploaded Images**.

In [6]:
if widgets is None:
    print("ipywidgets is unavailable. Use the local IMAGE_PATHS fallback cell below.")
else:
    uploader = widgets.FileUpload(accept="image/*", multiple=True)
    predict_button = widgets.Button(description="Predict Uploaded Images", button_style="success")
    output_box = widgets.Output()

    def uploaded_items(upload_widget):
        value = upload_widget.value
        if isinstance(value, dict):
            return [{"name": name, "content": data["content"]} for name, data in value.items()]
        return list(value)

    def on_predict_clicked(_):
        with output_box:
            output_box.clear_output()
            files = uploaded_items(uploader)
            if not files:
                print("Upload one or more image files first.")
                return

            results = []
            for file_info in files:
                image = Image.open(io.BytesIO(file_info["content"])).convert("RGB")
                prediction, confidence = predict_pil_image(image)
                results.append({
                    "file": file_info["name"],
                    "prediction": prediction,
                    "confidence": confidence,
                    "image": image.copy(),
                })

            show_prediction_results(results)

    predict_button.on_click(on_predict_clicked)
    display(widgets.VBox([uploader, predict_button, output_box]))

## Optional: Test Local Image Paths

Use this if the upload widget does not appear in your editor.

In [7]:
IMAGE_PATHS = [
    # r"../../archive/test/test/AppleCedarRust1.JPG",
    # r"../../archive/test/test/PotatoHealthy1.JPG",
]

if IMAGE_PATHS:
    results = []
    for image_path in IMAGE_PATHS:
        image = Image.open(image_path).convert("RGB")
        prediction, confidence = predict_pil_image(image)
        results.append({
            "file": os.path.basename(image_path),
            "prediction": prediction,
            "confidence": confidence,
            "image": image.copy(),
        })

    show_prediction_results(results)
else:
    print("Add one or more image paths to IMAGE_PATHS, then run this cell.")

Add one or more image paths to IMAGE_PATHS, then run this cell.
